In [ ]:
# Cell 1 - run information (required by the brief)
import importlib.metadata as _meta, subprocess, sys, random, numpy as np, torch

SEED = 266
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    # our GPU has 24 GB but the brief says everything must fit on 16 GB, so we hold ourselves to that
    _total = torch.cuda.get_device_properties(0).total_memory
    torch.cuda.set_per_process_memory_fraction(min(1.0, 16e9 / _total), 0)

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
BUILT_FROM = "808fa63"  # the repo commit this notebook was generated from


def _version(name):
    try:
        return _meta.version(name)
    except _meta.PackageNotFoundError:
        return "not installed yet"


print({"commit": commit or f"no git here, notebook built from {BUILT_FROM}", "built_from": BUILT_FROM, "python": sys.version.split()[0],
       "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})
print({name: _version(name) for name in ['pandas']})

# DATA 266 Lab 2, Part 3 results with the human forms (laptop, no GPU)

`p3_agents.ipynb` runs on the GPU and saves every scenario and red-team run to `reports/results/p3_runs.json`. The human pass/fail judgments happen after that run, so this small notebook recomputes the Part 3 results table from the saved runs plus `eval/human/p3_passfail_kavan.csv` and `p3_passfail_muhammed.csv`. It loads no models. Run it in the repo once both forms are filled in. Disagreements go in `eval/human/p3_resolved.csv`.

In [ ]:
%pip install -q pandas

In [ ]:
# Where are we running? Paths are relative, so the same notebook works on Colab, Kaggle or a local Jupyter.
import importlib.util, os
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB = importlib.util.find_spec("google.colab") is not None
here = Path.cwd().resolve()
# if the notebook happens to sit inside our repo, deliverables go straight into it
REPO = next((p for p in [here, *here.parents] if all((p / d).is_dir() for d in ("eval", "src", "notebooks"))), None)
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("/content") if ON_COLAB and Path("/content").is_dir() else (REPO or here)
# deliverables use the repo's folder layout, so the final ZIP unzips straight into the repo
OUT_ROOT = REPO or WORK / "lab2_repo_files"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("environment:", "kaggle" if ON_KAGGLE else "colab" if ON_COLAB else "local jupyter")
print("repo found:", REPO, "| working folder:", WORK, "| deliverables folder:", OUT_ROOT)
print("cuda:", torch.cuda.is_available())


def hf_token():
    """Hugging Face token from the HF_TOKEN variable, Colab or Kaggle secrets, or an earlier `huggingface-cli login`.
    It is never typed into the notebook."""
    token = os.environ.get("HF_TOKEN")
    if not token and ON_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
    if not token and ON_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None
    if not token:
        try:
            from huggingface_hub import get_token
            token = get_token()
        except Exception:
            token = None
    if token:
        os.environ["HF_TOKEN"] = token  # diffusers and huggingface_hub read it from here
    return token

In [ ]:
"""Part 3 results table: runs plus the human pass/fail forms -> reports/results/p3.json.

p3_agents.ipynb calls this right after the GPU run (human values print PENDING),
and p3_results.ipynb calls it again on a laptop once both forms are filled in.
Human values only ever come from the forms. Nothing here estimates them.
"""
from __future__ import annotations

import statistics
from pathlib import Path

import pandas as pd


def read_form(path: Path) -> pd.DataFrame:
    if not Path(path).exists():
        return pd.DataFrame(columns=["scenario_id", "mode", "pass", "critic_agree", "note"])
    form = pd.read_csv(path, dtype=str, keep_default_na=False)
    return form[form["pass"].isin(["0", "1"])]


def compute_results(runs: dict[str, pd.DataFrame], red: pd.DataFrame, human_dir: Path, raters=("kavan", "muhammed"), n_scenarios: int = 15) -> dict:
    human_dir = Path(human_dir)
    forms = {who: read_form(human_dir / f"p3_passfail_{who}.csv") for who in raters}
    complete = all(len(f) == 2 * n_scenarios for f in forms.values())
    results = {"baseline": {}, "final": {}, "red_team_attack_success": int(red.attack_succeeded.astype(bool).sum()), "red_team_cases": int(len(red))}
    for tag in ("baseline", "final"):
        results[tag].update({"median_seconds_per_request": float(statistics.median(runs[tag].seconds)),
                             "auto_checks_ok": int(runs[tag].auto_checks_ok.astype(bool).sum()),
                             "mean_revisions": float(runs[tag].revisions.mean())})
    if not complete:
        print("PENDING: human pass/fail forms are not complete, so scenarios passed, human agreement and critic agreement are not computed.")
        for tag in ("baseline", "final"):
            results[tag].update({"scenarios_passed": None, "human_agreement": None})
        results["final"]["critic_agreement"] = None
        return results

    a, b = raters
    merged = forms[a].merge(forms[b], on=["scenario_id", "mode"], suffixes=("_a", "_b"))
    merged["agree"] = merged.pass_a == merged.pass_b
    # disagreements are settled together and the agreed result goes in eval/human/p3_resolved.csv
    resolved_path = human_dir / "p3_resolved.csv"
    resolved = pd.read_csv(resolved_path, dtype=str) if resolved_path.exists() else None
    for tag in ("baseline", "final"):
        part = merged[merged["mode"] == tag]
        results[tag]["human_agreement"] = float(part.agree.mean())
        unresolved = part[~part.agree]
        if len(unresolved) and resolved is None:
            results[tag]["scenarios_passed"] = None
            print("PENDING:", tag, "has disagreements, add eval/human/p3_resolved.csv (scenario_id, mode, pass)")
        else:
            final_pass = part.set_index("scenario_id").pass_a.copy()
            if len(unresolved):
                final_pass.update(resolved[resolved["mode"] == tag].set_index("scenario_id")["pass"])
            results[tag]["scenarios_passed"] = int((final_pass == "1").sum())

    # critic agreement only counts scenarios where the critic actually made a decision
    decisions = runs["final"].set_index("id")
    made_ids = [i for i in decisions.index if len(decisions.loc[i, "critic_decisions"] or [])]
    votes = []
    for who in raters:
        f = forms[who][(forms[who]["mode"] == "final") & (forms[who].scenario_id.isin(made_ids))]
        votes += [v == "1" for v in f.critic_agree if v in ("0", "1")]
    results["final"]["critic_agreement"] = float(sum(votes) / len(votes)) if votes else None
    return results


def results_table(results: dict) -> pd.DataFrame:
    return pd.DataFrame({tag: results[tag] for tag in ("baseline", "final")}).astype(object).where(lambda d: d.notna(), "PENDING")

In [ ]:
import json
import pandas as pd

assert REPO, "run this notebook inside the repo, after unzipping the p3_agents ZIP into it"
saved = json.loads((REPO / "reports" / "results" / "p3_runs.json").read_text())
RUNS = {tag: pd.DataFrame(saved[tag]) for tag in ("baseline", "final")}
RED = pd.DataFrame(saved["red_team"])
results = compute_results(RUNS, RED, REPO / "eval" / "human")
(REPO / "reports" / "results" / "p3.json").write_text(json.dumps(results, indent=2))
print(json.dumps({k: results[k] for k in ("red_team_attack_success", "red_team_cases")}))
results_table(results)

In [ ]:
# Last cell - put this notebook's deliverables in one ZIP. It uses the repo's folder layout, so unzipping it
# at the repo root puts every file where it belongs. Original project files are only read, never changed.
import json as _json, shutil, zipfile

KEEP = ['reports/results/p3.json', 'eval/human']
export_dir = WORK / "export_p3_results"
if export_dir.exists():
    shutil.rmtree(export_dir)
(export_dir / "notebooks").mkdir(parents=True)

# 1) the executed notebook. On Colab the .ipynb is not on disk, so we ask the browser for it.
nb_target, saved = export_dir / "notebooks" / "p3_results.ipynb", False
if ON_COLAB:
    try:
        from google.colab import _message
        nb_target.write_text(_json.dumps(_message.blocking_request("get_ipynb", timeout_sec=120)["ipynb"]))
        saved = True
    except Exception as err:
        print("could not fetch the notebook from Colab:", type(err).__name__, err)
if not saved:
    for candidate in [Path("/kaggle/src/script.ipynb"), here / "p3_results.ipynb", *([REPO / "notebooks" / "p3_results.ipynb"] if REPO else [])]:
        if candidate.exists():
            shutil.copy(candidate, nb_target)
            saved = True
            break
if saved:
    print("notebook included. It has every output except this last cell, so also do File > Download .ipynb once this finishes.")
else:
    print("WARNING: notebook file not found. Do File > Download .ipynb and put it in notebooks/p3_results.ipynb in the repo.")

# 2) the deliverables, small files only (no raw data, model caches or secrets)
skip_parts = {"__pycache__", ".git", "chroma", "hf_cache", "data", "checkpoints"}
for rel in KEEP:
    source = OUT_ROOT / rel if (OUT_ROOT / rel).exists() else WORK / rel
    if source.is_file():
        (export_dir / rel).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(source, export_dir / rel)
    elif source.is_dir():
        for path in source.rglob("*"):
            if path.is_file() and not (skip_parts & set(path.relative_to(source).parts)) and path.stat().st_size < 300e6:
                target = export_dir / rel / path.relative_to(source)
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy(path, target)
    else:
        print("not found, skipped:", rel)

zip_path = WORK / "DATA266_Lab2_p3_results.zip"
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(export_dir.rglob("*")):
        if path.is_file():
            zf.write(path, path.relative_to(export_dir))

print(f"Created ZIP: {zip_path.resolve()} ({zip_path.stat().st_size / 1e6:.1f} MB). Unzip it at the repo root.")
with zipfile.ZipFile(zip_path) as zf:
    for info in zf.infolist():
        print(f"  {info.file_size / 1e6:8.2f} MB  {info.filename}")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    try:
        from IPython.display import FileLink, display
        display(FileLink(str(zip_path.relative_to(Path.cwd())) if zip_path.is_relative_to(Path.cwd()) else str(zip_path)))
    except Exception:
        pass
    print("If no download started, take the ZIP from the path above (file browser or output panel).")